In [ ]:
!pip install matplotlib scikit-learn

In [3]:
import sys
import os
# Force Jupyter to look in both the root and the scripts folder!
sys.path.append(os.path.abspath('./scripts'))
sys.path.append(os.path.abspath('.'))

import sqlite3
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Now it will find them!
try:
    from database import cipher_suite
except ModuleNotFoundError:
    from scripts.database import cipher_suite

try:
    from adaptive import KnowledgeTracer
except ModuleNotFoundError:
    from scripts.adaptive import KnowledgeTracer

# 1. Read the Encrypted Data
conn = sqlite3.connect('tutor_progress.db')
c = conn.cursor()
c.execute("SELECT encrypted_skill, encrypted_correct FROM progress")
rows = c.fetchall()
conn.close()

# 2. Decrypt the history
history = []
for enc_skill, enc_correct in rows:
    skill = cipher_suite.decrypt(enc_skill).decode()
    correct = int(cipher_suite.decrypt(enc_correct).decode())
    history.append((skill, correct))

print(f"Loaded {len(history)} real interactions from the database.")

Loaded 4 real interactions from the database.


In [5]:
# Initialize Models
kt = KnowledgeTracer()
elo_rating = 1200
item_difficulty = 1200
K = 32

actuals = []
bkt_preds = []
elo_preds = []

# Replay history step-by-step
for skill, actual in history:
    actuals.append(actual)
    
    # --- 1. BKT Prediction ---
    s = kt.skills.get(skill, kt.skills["counting"])
    pk = s["p_know"]
    p_correct_bkt = (pk * (1 - s["p_slip"])) + ((1 - pk) * s["p_guess"])
    bkt_preds.append(p_correct_bkt)
    kt.update(skill, actual)
    
    # --- 2. Elo Prediction ---
    p_correct_elo = 1 / (1 + 10**((item_difficulty - elo_rating) / 400))
    elo_preds.append(p_correct_elo)
    elo_rating = elo_rating + K * (actual - p_correct_elo)

# Plotting the Graph
if len(actuals) < 5:
    print("⚠️ You need more data! Go play at least 10-15 questions in demo.py to draw a smooth curve.")
else:
    # Calculate ROC metrics
    fpr_elo, tpr_elo, _ = roc_curve(actuals, elo_preds)
    auc_elo = auc(fpr_elo, tpr_elo)

    fpr_bkt, tpr_bkt, _ = roc_curve(actuals, bkt_preds)
    auc_bkt = auc(fpr_bkt, tpr_bkt)

    # Plot the Graph
    plt.figure(figsize=(8, 6))
    plt.plot(fpr_elo, tpr_elo, label=f'Elo Baseline (AUC = {auc_elo:.3f})', linestyle='--', color='orange')
    plt.plot(fpr_bkt, tpr_bkt, label=f'BKT Model (AUC = {auc_bkt:.3f})', linewidth=2, color='blue')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.500)')

    plt.title('Knowledge Tracing: BKT vs. Elo on Real Learner Data')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    
    # Save image for your GitHub Repo
    plt.savefig('kt_roc_curve.png')
    plt.show()

⚠️ You need more data! Go play at least 10-15 questions in demo.py to draw a smooth curve.
